<a href="https://colab.research.google.com/github/MichalSlowakiewicz/Visual-Recognition/blob/master/LAB_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Spatial Transformer
In this lab scenario, we're implementing a differentiable module that allows networks to perform spatial transformations on input images (or feature maps). For details, you can refer to [the paper](https://arxiv.org/abs/1506.02025).


## Imports

In [ ]:
%pip install lightning --quiet

In [ ]:
from pathlib import Path
from typing import cast

import lightning as L
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.datasets
from lightning.pytorch.loggers import TensorBoardLogger
from torch import Tensor
from torch.utils.data import DataLoader
from torchvision.transforms import v2


## Dataset
For training, we are going to use the MNIST dataset, with a strong RandomAffine augmentation.

In [ ]:
BATCH_SIZE = 128
IMAGE_HEIGHT, IMAGE_WIDTH = 28, 28

In [ ]:
train_transforms = v2.Compose(
    [
        v2.ToImage(),
        v2.RandomAffine(
            degrees=45,
            translate=(0.25, 0.25),
            scale=(0.5, 1.0),
            interpolation=v2.InterpolationMode.BILINEAR,
        ),
        v2.ToDtype(torch.float32, scale=True),
    ]
)

test_transforms = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])

to_pil_image = v2.Compose(
    [
        # v2.Normalize([-m / s for m, s in zip(mean, std)], [1 / s for s in std]),
        v2.ToPILImage(),
        v2.Resize(128, interpolation=v2.InterpolationMode.NEAREST),
    ]
)


dataset_path = Path("./data/MNIST")
train_dataset = torchvision.datasets.MNIST(
    root=dataset_path, train=True, transform=train_transforms, download=True
)
test_dataset = torchvision.datasets.MNIST(
    root=dataset_path, train=False, transform=test_transforms, download=True
)
CLASSES = train_dataset.classes

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)

In [ ]:
def show_images(
    images: Tensor,
    labels: Tensor | None = None,
    n_rows: int = 1,
    n_cols: int = -1,
    title: str = "",
) -> None:
    if n_cols == -1:
        n_cols = len(images) // n_rows
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2 * n_cols, 2 * n_rows))
    fig.set_layout_engine("compressed", h_pad=0.1, w_pad=0, hspace=0, wspace=0)
    if n_rows == 1 == n_cols:
        axes = np.array([[axes]])
    elif n_rows == 1:
        axes = axes[np.newaxis, :]
    elif n_cols == 1:
        axes = axes[:, np.newaxis]
    for i, img in enumerate(images[: n_rows * n_cols]):
        r, c = divmod(i, n_cols)
        axes[r, c].imshow(to_pil_image(img))
        axes[r, c].axis("off")
        if labels is not None:
            axes[r, c].set_title(CLASSES[labels[i]])

    # Title left of first row.
    axes[0, 0].axis("on")
    axes[0, 0].xaxis.set_ticks([])
    axes[0, 0].yaxis.set_ticks([])
    axes[0, 0].set_ylabel(title, rotation=90)


image_batch, label_batch = next(iter(train_loader))
show_images(image_batch[:8], label_batch[:8])

## Spatial Tools
The ***Spatial Transformer*** (unrelated to transformers) is an architecure that
aims to learn visual features that are invariant to transformations like rotation, scale change and shifts.

The basic version consists of:
* A localization network: given an input (source) image or feature map $S \in \mathbb{R}^{C \times H_{\text{src}} \times W_{\text{src}}}$, it predicts parameters $\theta$ of a transformation.
* A classification network: given a transformed image/feature map $T \in \mathbb{R}^{C \times H_{\text{tgt}} \times W_{\text{tgt}}}$ it produces a class prediction (`num_classes` logits).

We will apply the transformation in a way that is differentiable with respect to the parameters $\theta$ (and the input $S$, in case it is a feature map).

For this version, we consider affine 2D transformations, parameterized by $\theta \in \mathbb{R}^{2 \times 3}$:
$$
    \left(\begin{array}{c}
    y_{\text{src}}\\
    x_{\text{src}}
    \end{array}\right)
    =
    \left(\begin{array}{ccc}
    \theta_1 & \theta_2 & \theta_3\\
    \theta_4 & \theta_5 & \theta_6
    \end{array}\right)
    \left(\begin{array}{c}
    y_{\text{tgt}}\\
    x_{\text{tgt}}\\
    1
    \end{array}\right)
    =
    \left(\begin{array}{ccc}
    \theta_1 & \theta_2\\
    \theta_4 & \theta_5
    \end{array}\right)
    \left(\begin{array}{c}
    y_{\text{tgt}}\\
    x_{\text{tgt}}
    \end{array}\right)    
    +
    \left(\begin{array}{c}
    \theta_3\\
    \theta_6    
    \end{array}\right)    
$$
where:
* $(y_{\text{tgt}}, x_{\text{tgt}}) \in [-1,1] \times [-1,1]$ are normalized coordinates in the output image,
* $(y_{\text{src}}, x_{\text{src}}) \in [-1,1] \times [-1,1]$ are normalized coordinates in the input image.

(Note that we use $y$ before $x$ to be consistent with the convention of indexing images as $H$ before $W$.)
  
We will denoted unnormalized coordinates as:
* $(h_{\text{tgt}}, w_{\text{tgt}}) \in \texttt{range}(H_{\text{tgt}}) \times \texttt{range}(W_{\text{tgt}})$ – we'll only consider integers here,
* $(h_{\text{src}}, w_{\text{src}}) \in \texttt{range}(H_{\text{src}}) \times \texttt{range}(W_{\text{src}})$ – these can be real numbers.

The color ($\mathbb{R}^C$) of each pixel $(h_{\text{tgt}}, w_{\text{tgt}})$ of $T$ can be obtained from colors of $S$ around $(h_{\text{src}}, w_{\text{src}})$ in many ways – we will use bilinear interpolation.
That is, the value will be a weighted average of values of pixels in $S$ at coordinates:
* $\lfloor h_{\text{src}} \rfloor, \lfloor w_{\text{src}} \rfloor$
* $\lfloor h_{\text{src}} \rfloor, \lceil w_{\text{src}} \rceil$
* $\lceil h_{\text{src}} \rceil, \lfloor w_{\text{src}} \rfloor$
* $\lceil h_{\text{src}} \rceil, \lceil w_{\text{src}} \rceil$

We'll use 0 as the color of pixels in $S$ that are outside of the image bounds.

(Note that here we interpret $(h_{\text{src}}, w_{\text{src}})=(0,0)$ as the center of the top-left pixel, and $(H_{\text{src}}-1, W_{\text{src}}-1)$ as the center of the bottom-right pixel.
In other contexts, it can be more natural to interpret $(h_{\text{src}}, w_{\text{src}})=(0,0)$ as the top-left corner of the top-left pixel, $(\frac{1}{2}, \frac{1}{2})$ as its center and $(H_{\text{src}}, W_{\text{src}})$ as the bottom-right corner of the bottom-right pixel.)

Before you start the implementation one more question.  
**Why do we use bilinear interpolation here instead of just picking the nearest pixel** $S[\texttt{round}(h_{\text{src}}), \texttt{round}(w_{\text{src}})]$?

**Fill in the names** for the following affine transformations: diagonal flip, identity, left-right flip, rotate 90, scale&translate, translate, upside-down flip.

In [ ]:
# TODO {
transforms = {
    "": torch.tensor([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0]]),
    "": torch.tensor([[1.0, 0.0, 0.5], [0.0, 1.0, 0.25]]),
    "": torch.tensor([[0.0, 1.0, 0.0], [1.0, 0.0, 0.0]]),
    "": torch.tensor([[-1.0, 0.0, 0.0], [0.0, 1.0, 0.0]]),
    "": torch.tensor([[1.0, 0.0, 0.0], [0.0, -1.0, 0.0]]),
    "": torch.tensor([[0.0, -1.0, 0.0], [1.0, 0.0, 0.0]]),
    "": torch.tensor([[1.5, 0.0, 0.75], [0.0, 1.5, 0.75]]),
}
# }

##  get_sampling_grid
Finish the implementation of `get_sampling_grid` according to the docstring.
**Please do not use built-in functions like `affine_grid` or `grid_sample`**

In [ ]:
def get_sampling_grid(theta: Tensor, H_tgt: int, W_tgt: int) -> Tensor:
    """
    Given parameters of B affine transformations theta of shape (B, 2, 3).
    returns a sampling_grid of shape (B, H_tgt, W_tgt, 2).

    `sampling_grid[b, h_tgt, w_tgt]` should be the (y_src,x_src) coordinates
    in the source image resulting from:
    * normalize the pixel coordinates (h_tgt, w_tgt) to the range [-1, 1],
    * applying the b-th transformation to the normalized pixel coordinates (y_tgt, x_tgt).

    No clipping is performed – the results may lay outside the [-1, 1] range.
    """

    B = theta.shape[0]
    assert theta.shape == (B, 2, 3), f"Expected ({B=}, 2, 3), got {theta.shape=}"

    # TODO {
    # }

    assert sampling_grid.shape == (B, H_tgt, W_tgt, 2)
    # Make it contiguous, as required by `view_as_complex()`.
    return sampling_grid.contiguous()

#### Tests

In [ ]:
sampling_grid1 = get_sampling_grid(
    transforms["identity"].unsqueeze(0), H_tgt=3, W_tgt=3
).squeeze(0)
print("sampling_grid1 (identity)")
print(torch.view_as_complex(sampling_grid1))

assert torch.allclose(
    sampling_grid1,
    torch.tensor(
        [
            [[-1, -1], [-1, 0], [-1, 1]],
            [[0, -1], [0, 0], [0, 1]],
            [[1, -1], [1, 0], [1, 1]],
        ],
        dtype=torch.float32,
    ),
)

In [ ]:
sampling_grid2 = get_sampling_grid(
    transforms["translate"].unsqueeze(0), H_tgt=3, W_tgt=3
).squeeze(0)
print("sampling_grid2 (translate)")
print(torch.view_as_complex(sampling_grid2))
assert torch.allclose(sampling_grid2, sampling_grid1 + torch.tensor([[[0.5, 0.25]]]))

In [ ]:
sampling_grid3 = get_sampling_grid(
    transforms["diagonal flip"].unsqueeze(0), H_tgt=3, W_tgt=3
).squeeze(0)
print("sampling_grid3 (diagonal flip)")
print(torch.view_as_complex(sampling_grid3))
assert torch.allclose(
    torch.view_as_complex(sampling_grid3), torch.view_as_complex(sampling_grid1).T
)

In [ ]:
sampling_grid4 = get_sampling_grid(
    transforms["identity"].unsqueeze(0).expand(7, -1, -1), H_tgt=3, W_tgt=3
)
assert sampling_grid4.shape == (7, 3, 3, 2)
print("sampling_grid4 (batch of identities)")
assert torch.allclose(
    sampling_grid4,
    sampling_grid1.unsqueeze(0).expand(7, -1, -1, -1),
)

## apply_sampling_grid
Finish the implementation of `apply_sampling_grid` according to the docstring.

In [ ]:
def apply_sampling_grid(source_image: Tensor, sampling_grid: Tensor) -> Tensor:
    """
    Sample a target image from the source_image using the sampling_grid.

    Args:
    - source_image: shape (B, C, H_src, W_src).
    - sampling_grid: shape (B, H_tgt, W_tgt, 2) of normalized ([-1,1]) coordinates in the source image.

    Returns: image of shape (B, C, H_tgt, W_tgt).

    Uses bilinear interpolation and zero padding.
    """

    B, C, H_src, W_src = source_image.shape
    _, H_tgt, W_tgt, _ = sampling_grid.shape
    assert sampling_grid.shape == (B, H_tgt, W_tgt, 2)
    # TODO {
    # }

    assert transformed_image.shape == (B, C, H_tgt, W_tgt)
    return transformed_image

Let's check whether this works as expected.

In [ ]:
def visualize_transformation(
    title: str,
    transform_matrix: Tensor,  # shape (B, 2, 3) or (1, 2, 3)
    image_batch: Tensor,  # shape (B, C, H_src, W_src)
    label_batch: Tensor,
    H_tgt: int | None = None,
    W_tgt: int | None = None,
) -> None:
    B, C, H_src, W_src = image_batch.shape
    H_tgt = H_tgt or H_src
    W_tgt = W_tgt or W_src
    transform_matrix = transform_matrix.expand(B, -1, -1)  # broadcast to batch size.
    sampling_grid = get_sampling_grid(transform_matrix, H_tgt=H_tgt, W_tgt=W_tgt)
    res = apply_sampling_grid(image_batch, sampling_grid)
    show_images(res, label_batch, title=title)


image_batch, label_batch = next(iter(test_loader))
for title, transform in transforms.items():
    visualize_transformation(
        title, transform.unsqueeze(0), image_batch[:8], label_batch[:8]
    )

## Network

Finish the implementation of `NetWithSpatialTransformer.locate_and_transform()`.

In [ ]:
class LocNet(nn.Module):
    """Localisation network."""

    def __init__(self, in_channels: int = 1) -> None:
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(in_channels=in_channels, out_channels=4, kernel_size=3),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.ReLU(),
            nn.Conv2d(in_channels=4, out_channels=8, kernel_size=3),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.ReLU(),
            nn.Flatten(),
            nn.Dropout(p=0.2),
            nn.Linear(200, 32),
            nn.ReLU(),
            nn.Linear(32, 6),
        )

        # Initialize so that the output is always an identity transformation.
        last = cast(nn.Linear, self.layers[-1])
        last.weight.data.zero_()
        last.bias.data.copy_(torch.tensor([1.0, 0.0, 0.0, 0.0, 1.0, 0.0]))

    def forward(self, images: Tensor) -> Tensor:
        """
        Input: images of shape (B, C, H, W).
        Output: affine transformation parameters theta of shape (B, 2, 3).
        """
        assert len(images.shape) == 4
        theta = self.layers(images)
        return theta.view(-1, 2, 3)


class NetWithSpatialTransformer(nn.Module):
    def __init__(
        self,
        in_channels: int = 1,
        H_tgt: int = IMAGE_HEIGHT,
        W_tgt: int = IMAGE_WIDTH,
        skip_loc_net: bool = False,
    ) -> None:
        super().__init__()
        self.H_tgt = H_tgt
        self.W_tgt = W_tgt
        self.skip_loc_net = skip_loc_net

        if not self.skip_loc_net:
            self.loc_net = LocNet(in_channels=in_channels)

        # MNIST is a very simple dataset.
        # To strongly motivate the use of the spatial transformer, we use a simple model here.
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(),
            nn.Linear(in_channels * H_tgt * W_tgt, 32),
            nn.ReLU(),
            nn.Linear(32, len(CLASSES)),
        )

    def locate_and_transform(self, images: Tensor) -> Tensor:
        """
        Input: images of shape (B, C, H_src, W_src).
        Output: transformed images of shape (B, C, H_tgt, W_tgt).
        """
        assert len(images.shape) == 4
        # TODO {
        # }
        return sampled

    def forward(self, images: Tensor) -> Tensor:
        if self.skip_loc_net:
            sampled = F.interpolate(
                images, size=(self.H_tgt, self.W_tgt), mode="bilinear"
            )
        else:
            sampled = self.locate_and_transform(images)
        logits = self.classifier(sampled)
        return logits

In [ ]:
net = NetWithSpatialTransformer()
assert net(image_batch).shape == (len(image_batch), len(CLASSES))

## Training

Note that gradients will be computed over the classifier's params, over the sampled image, then over the sampling grid, then over the parameters $\theta$, and finally over the parameters of the localization network.

In [ ]:
class SpatialTransformerLightningModule(L.LightningModule):
    def __init__(self, model: nn.Module) -> None:
        super().__init__()
        self.model = model

    def train_dataloader(self) -> DataLoader:
        return train_loader

    def val_dataloader(self) -> DataLoader:
        return test_loader

    def training_step(self, batch: tuple[Tensor, Tensor], batch_idx: int) -> Tensor:
        img, lbl = batch
        logits = self.model(img)
        loss = F.cross_entropy(logits, lbl)
        pred = torch.argmax(logits, dim=-1)
        accuracy = (pred == lbl).float().mean()

        self.log("train/loss", loss.detach())
        self.log("train/acc", accuracy.detach(), prog_bar=True)

        return loss

    def validation_step(self, batch: tuple[Tensor, Tensor], batch_idx: int) -> None:
        img, lbl = batch
        logits = self.model(img)
        pred = torch.argmax(logits, dim=-1)
        accuracy = (pred == lbl).float().mean()

        self.log("test/acc", accuracy.detach(), on_epoch=True, prog_bar=True)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.model.parameters())
        return optimizer

In [ ]:
%load_ext tensorboard
!mkdir -p runs/spatial_transformer
%tensorboard --logdir runs/spatial_transformer

In [ ]:
net = NetWithSpatialTransformer()

lightning_module = SpatialTransformerLightningModule(model=net)

trainer = L.Trainer(
    logger=TensorBoardLogger("runs/spatial_transformer", name="spatial"),
    max_epochs=2,
    check_val_every_n_epoch=1,
)

trainer.fit(lightning_module)

## Inspection
Let's check what our spatial transformed learned.

First on unaltered images:

In [ ]:
net.eval()
net.to("cpu");

In [ ]:
image_batch, label_batch = next(iter(test_loader))
show_images(image_batch[:8], label_batch[:8])
show_images(net.locate_and_transform(image_batch[:8]), label_batch[:8])

Then on altered (augmented) images:

In [ ]:
image_batch, label_batch = next(iter(train_loader))
show_images(image_batch[:8], label_batch[:8])
show_images(net.locate_and_transform(image_batch[:8]), label_batch[:8])

## Comparison with baseline
(Note that the training is short and the baseline has far fewer parameters).

In [ ]:
lightning_module = SpatialTransformerLightningModule(
    model=NetWithSpatialTransformer(skip_loc_net=True)
)

trainer = L.Trainer(
    logger=TensorBoardLogger("runs/spatial_transformer", name="baseline"),
    max_epochs=2,
    check_val_every_n_epoch=1,
)

trainer.fit(lightning_module)